# Day 09. Exercise 00
# Regularization

## 0. Imports

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [29]:
df = pd.read_csv('../data/dayofweek.csv')
print(df.shape)
df.head()

(1686, 44)


,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.692958,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.661055,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [30]:
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Распределение значений зависимой переменной:\n{y.value_counts().sort_index()}")

X shape: (1686, 43), y shape: (1686,)
Распределение значений зависимой переменной:
dayofweek
0    136
1    274
2    149
3    396
4    104
5    271
6    356
Name: count, dtype: int64


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (1348, 43), X_test: (338, 43)
y_train: (1348,), y_test: (338,)


## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [32]:
def evaluate_model_cv(model, X_train, y_train, n_splits=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=False)
    train_scores = []
    valid_scores = []

    for train_idx, valid_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        train_acc = accuracy_score(y_tr, model.predict(X_tr))
        valid_acc = accuracy_score(y_val, model.predict(X_val))
        train_scores.append(train_acc)
        valid_scores.append(valid_acc)
        print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")

    avg = np.mean(valid_scores)
    std = np.std(valid_scores)
    print(f"Average accuracy on crossval is {avg:.5f}")
    print(f"Std is {std:.5f}")
    return avg, std

In [33]:
%%time
logreg_base = LogisticRegression(random_state=21, fit_intercept=False)
logreg_avg, logreg_std = evaluate_model_cv(logreg_base, X_train, y_train, n_splits=10)

train -  0.62819   |   valid -  0.59259
train -  0.64716   |   valid -  0.62963
train -  0.63479   |   valid -  0.57037
train -  0.65540   |   valid -  0.61481
train -  0.63314   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63591   |   valid -  0.62687
Average accuracy on crossval is 0.60239
Std is 0.02852
CPU times: user 1.51 s, sys: 941 μs, total: 1.51 s
Wall time: 158 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [34]:
%%time
print('Penalty: L1')
logreg_l1 = LogisticRegression(random_state=21, fit_intercept=False, l1_ratio = 1, solver='saga', max_iter=5000)
l1_avg, l1_std = evaluate_model_cv(logreg_l1, X_train, y_train, n_splits=10)

Penalty: L1
train -  0.63726   |   valid -  0.58519
train -  0.64221   |   valid -  0.61481
train -  0.62984   |   valid -  0.55556
train -  0.64386   |   valid -  0.60000
train -  0.63232   |   valid -  0.57778
train -  0.63644   |   valid -  0.57778
train -  0.63644   |   valid -  0.65926
train -  0.65622   |   valid -  0.57778
train -  0.64580   |   valid -  0.58955
train -  0.63839   |   valid -  0.62687
Average accuracy on crossval is 0.59646
Std is 0.02848
CPU times: user 2.73 s, sys: 2.04 ms, total: 2.73 s
Wall time: 2.2 s


In [35]:
%%time
print('Penalty: L2')
logreg_l2 = LogisticRegression(random_state=21, fit_intercept=False, l1_ratio = 0, solver='lbfgs', max_iter=5000)
l2_avg, l2_std = evaluate_model_cv(logreg_l2, X_train, y_train, n_splits=10)

Penalty: L2
train -  0.62819   |   valid -  0.59259
train -  0.64716   |   valid -  0.62963
train -  0.63479   |   valid -  0.57037
train -  0.65540   |   valid -  0.61481
train -  0.63314   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63591   |   valid -  0.62687
Average accuracy on crossval is 0.60239
Std is 0.02852
CPU times: user 1.25 s, sys: 0 ns, total: 1.25 s
Wall time: 130 ms


In [36]:
%%time
print('Penalty: None')
logreg_none = LogisticRegression(random_state=21, fit_intercept=False, C=np.inf, solver='lbfgs', max_iter=5000)
none_avg, none_std = evaluate_model_cv(logreg_none, X_train, y_train, n_splits=10)

Penalty: None
train -  0.66612   |   valid -  0.63704
train -  0.65622   |   valid -  0.65926
train -  0.66694   |   valid -  0.57778
train -  0.66529   |   valid -  0.62963
train -  0.66777   |   valid -  0.61481
train -  0.65870   |   valid -  0.57778
train -  0.64963   |   valid -  0.69630
train -  0.68508   |   valid -  0.61481
train -  0.66392   |   valid -  0.62687
train -  0.65733   |   valid -  0.60448
Average accuracy on crossval is 0.62388
Std is 0.03392
CPU times: user 3.04 s, sys: 2.96 ms, total: 3.04 s
Wall time: 304 ms


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [37]:
%%time
svm_base = SVC(probability=True, kernel='linear', random_state=21)
svm_avg, svm_std = evaluate_model_cv(svm_base, X_train, y_train, n_splits=10)

train -  0.70486   |   valid -  0.65926
train -  0.69662   |   valid -  0.75556
train -  0.69415   |   valid -  0.62222
train -  0.70239   |   valid -  0.65185
train -  0.69085   |   valid -  0.65185
train -  0.68920   |   valid -  0.64444
train -  0.69250   |   valid -  0.72593
train -  0.70074   |   valid -  0.62222
train -  0.69605   |   valid -  0.61940
train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: user 2.08 s, sys: 2.05 ms, total: 2.08 s
Wall time: 1.55 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [38]:
%%time
print('SVM C=0.1')
svm_01 = SVC(probability=True, kernel='linear', random_state=21, C=0.1)
svm_01_avg, svm_01_std = evaluate_model_cv(svm_01, X_train, y_train, n_splits=10)

SVM C=0.1
train -  0.58120   |   valid -  0.55556
train -  0.57543   |   valid -  0.56296
train -  0.57378   |   valid -  0.57037
train -  0.59275   |   valid -  0.57037
train -  0.58120   |   valid -  0.54815
train -  0.57955   |   valid -  0.54815
train -  0.57296   |   valid -  0.61481
train -  0.59192   |   valid -  0.54815
train -  0.59967   |   valid -  0.52985
train -  0.57825   |   valid -  0.57463
Average accuracy on crossval is 0.56230
Std is 0.02177
CPU times: user 1.53 s, sys: 2.01 ms, total: 1.54 s
Wall time: 1.53 s


In [39]:
%%time
print('SVM C=10')
svm_10 = SVC(probability=True, kernel='linear', random_state=21, C=10)
svm_10_avg, svm_10_std = evaluate_model_cv(svm_10, X_train, y_train, n_splits=10)

SVM C=10
train -  0.75021   |   valid -  0.72593
train -  0.77741   |   valid -  0.82963
train -  0.78566   |   valid -  0.68148
train -  0.76834   |   valid -  0.73333
train -  0.75185   |   valid -  0.77778
train -  0.75598   |   valid -  0.68889
train -  0.76257   |   valid -  0.74074
train -  0.77411   |   valid -  0.68889
train -  0.78254   |   valid -  0.71642
train -  0.78418   |   valid -  0.69403
Average accuracy on crossval is 0.72771
Std is 0.04417
CPU times: user 2.21 s, sys: 35 μs, total: 2.21 s
Wall time: 2.21 s


In [40]:
%%time
print('SVM C=0.01')
svm_001 = SVC(probability=True, kernel='linear', random_state=21, C=0.01)
svm_001_avg, svm_001_std = evaluate_model_cv(svm_001, X_train, y_train, n_splits=10)

SVM C=0.01
train -  0.37923   |   valid -  0.40000
train -  0.37923   |   valid -  0.40000
train -  0.38417   |   valid -  0.35556
train -  0.35449   |   valid -  0.36296
train -  0.38252   |   valid -  0.37037
train -  0.38087   |   valid -  0.38519
train -  0.37923   |   valid -  0.40000
train -  0.38252   |   valid -  0.37037
train -  0.38468   |   valid -  0.35075
train -  0.38386   |   valid -  0.35821
Average accuracy on crossval is 0.37534
Std is 0.01848
CPU times: user 1.72 s, sys: 3.01 ms, total: 1.72 s
Wall time: 1.72 s


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [41]:
%%time
tree_base = DecisionTreeClassifier(max_depth=10, random_state=21)
tree_avg, tree_std = evaluate_model_cv(tree_base, X_train, y_train, n_splits=10)

train -  0.81039   |   valid -  0.74074
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.76296
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.75373
train -  0.80478   |   valid -  0.68657
Average accuracy on crossval is 0.72551
Std is 0.03562
CPU times: user 49.8 ms, sys: 8 μs, total: 49.8 ms
Wall time: 49.2 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [42]:
%%time
print('Tree max_depth=5')
tree_5 = DecisionTreeClassifier(max_depth=5, random_state=21)
tree_5_avg, tree_5_std = evaluate_model_cv(tree_5, X_train, y_train, n_splits=10)

Tree max_depth=5
train -  0.59522   |   valid -  0.53333
train -  0.56307   |   valid -  0.53333
train -  0.60181   |   valid -  0.55556
train -  0.59604   |   valid -  0.57037
train -  0.60264   |   valid -  0.57778
train -  0.57955   |   valid -  0.53333
train -  0.58368   |   valid -  0.54815
train -  0.59275   |   valid -  0.51111
train -  0.58237   |   valid -  0.56716
train -  0.60132   |   valid -  0.50000
Average accuracy on crossval is 0.54301
Std is 0.02423
CPU times: user 47 ms, sys: 1e+03 ns, total: 47 ms
Wall time: 46.4 ms


In [43]:
%%time
print('Tree max_depth=15')
tree_15 = DecisionTreeClassifier(max_depth=15, random_state=21)
tree_15_avg, tree_15_std = evaluate_model_cv(tree_15, X_train, y_train, n_splits=10)

Tree max_depth=15
train -  0.95796   |   valid -  0.82222
train -  0.93075   |   valid -  0.83704
train -  0.95631   |   valid -  0.83704
train -  0.95301   |   valid -  0.86667
train -  0.95136   |   valid -  0.88889
train -  0.94724   |   valid -  0.82222
train -  0.95466   |   valid -  0.90370
train -  0.94971   |   valid -  0.87407
train -  0.95305   |   valid -  0.83582
train -  0.94316   |   valid -  0.85821
Average accuracy on crossval is 0.85459
Std is 0.02682
CPU times: user 54.7 ms, sys: 997 μs, total: 55.7 ms
Wall time: 54.6 ms


In [44]:
%%time
print('Tree max_depth=20')
tree_20 = DecisionTreeClassifier(max_depth=20, random_state=21)
tree_20_avg, tree_20_std = evaluate_model_cv(tree_20, X_train, y_train, n_splits=10)

Tree max_depth=20
train -  0.98846   |   valid -  0.86667
train -  0.99011   |   valid -  0.91111
train -  0.98681   |   valid -  0.85926
train -  0.98763   |   valid -  0.91111
train -  0.98928   |   valid -  0.88148
train -  0.98186   |   valid -  0.85926
train -  0.98846   |   valid -  0.91852
train -  0.99176   |   valid -  0.89630
train -  0.99094   |   valid -  0.88060
train -  0.98847   |   valid -  0.88060
Average accuracy on crossval is 0.88649
Std is 0.02075
CPU times: user 54.6 ms, sys: 2 ms, total: 56.6 ms
Wall time: 55.3 ms


In [45]:
%%time
print('Tree max_depth=10, min_samples_split=10, min_samples_leaf=5')
tree_opt = DecisionTreeClassifier(max_depth=20, min_samples_split=10, min_samples_leaf=5, random_state=21)
tree_opt_avg, tree_opt_std = evaluate_model_cv(tree_opt, X_train, y_train, n_splits=10)

Tree max_depth=10, min_samples_split=10, min_samples_leaf=5
train -  0.86645   |   valid -  0.78519
train -  0.86480   |   valid -  0.74074
train -  0.87634   |   valid -  0.72593
train -  0.86645   |   valid -  0.82222
train -  0.87387   |   valid -  0.80741
train -  0.86150   |   valid -  0.80000
train -  0.86645   |   valid -  0.82222
train -  0.86727   |   valid -  0.74074
train -  0.86079   |   valid -  0.72388
train -  0.86573   |   valid -  0.76866
Average accuracy on crossval is 0.77370
Std is 0.03692
CPU times: user 52.2 ms, sys: 1.98 ms, total: 54.2 ms
Wall time: 52.9 ms


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [46]:
%%time
rf_base = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
rf_avg, rf_std = evaluate_model_cv(rf_base, X_train, y_train, n_splits=10)

train -  0.96455   |   valid -  0.88148
train -  0.96208   |   valid -  0.91852
train -  0.96785   |   valid -  0.86667
train -  0.96455   |   valid -  0.89630
train -  0.96538   |   valid -  0.91111
train -  0.96538   |   valid -  0.88148
train -  0.97115   |   valid -  0.91852
train -  0.96867   |   valid -  0.85185
train -  0.97364   |   valid -  0.88060
train -  0.97941   |   valid -  0.86567
Average accuracy on crossval is 0.88722
Std is 0.02204
CPU times: user 577 ms, sys: 4.02 ms, total: 581 ms
Wall time: 577 ms


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [47]:
%%time
print('RF max_depth=20, n_estimators=100')
rf_20_100 = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=21)
rf_20_100_avg, rf_20_100_std = evaluate_model_cv(rf_20_100, X_train, y_train, n_splits=10)

RF max_depth=20, n_estimators=100
train -  0.99670   |   valid -  0.89630
train -  0.99753   |   valid -  0.94815
train -  0.99670   |   valid -  0.89630
train -  0.99753   |   valid -  0.92593
train -  0.99670   |   valid -  0.91111
train -  0.99670   |   valid -  0.88889
train -  0.99670   |   valid -  0.91852
train -  0.99670   |   valid -  0.91111
train -  0.99918   |   valid -  0.92537
train -  0.99753   |   valid -  0.88060
Average accuracy on crossval is 0.91023
Std is 0.01925
CPU times: user 1.14 s, sys: 3.01 ms, total: 1.15 s
Wall time: 1.14 s


In [48]:
%%time
print('RF max_depth=22, n_estimators=50')
rf_22_50 = RandomForestClassifier(n_estimators=50, max_depth=22, random_state=21)
rf_22_50_avg, rf_22_50_std = evaluate_model_cv(rf_22_50, X_train, y_train, n_splits=10)

RF max_depth=22, n_estimators=50
train -  0.99835   |   valid -  0.89630
train -  0.99918   |   valid -  0.94815
train -  0.99918   |   valid -  0.88889
train -  0.99835   |   valid -  0.93333
train -  0.99753   |   valid -  0.91852
train -  0.99670   |   valid -  0.88889
train -  0.99835   |   valid -  0.91852
train -  0.99835   |   valid -  0.89630
train -  0.99918   |   valid -  0.94030
train -  0.99835   |   valid -  0.88060
Average accuracy on crossval is 0.91098
Std is 0.02277
CPU times: user 613 ms, sys: 5.99 ms, total: 619 ms
Wall time: 616 ms


In [49]:
%%time
print('RF max_depth=10, n_estimators=50')
rf_10_50 = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=21)
rf_10_50_avg, rf_10_50_std = evaluate_model_cv(rf_10_50, X_train, y_train, n_splits=10)

RF max_depth=10, n_estimators=50
train -  0.85408   |   valid -  0.75556
train -  0.85903   |   valid -  0.82963
train -  0.89777   |   valid -  0.80741
train -  0.90107   |   valid -  0.82222
train -  0.88376   |   valid -  0.85185
train -  0.87799   |   valid -  0.75556
train -  0.87222   |   valid -  0.82963
train -  0.86480   |   valid -  0.74074
train -  0.89456   |   valid -  0.82090
train -  0.87644   |   valid -  0.73881
Average accuracy on crossval is 0.79523
Std is 0.04051
CPU times: user 508 ms, sys: 3.99 ms, total: 512 ms
Wall time: 510 ms


In [50]:
%%time
print('RF max_depth=22, n_estimators=100, min_samples_leaf=2')
rf_best = RandomForestClassifier(n_estimators=100, max_depth=22, min_samples_leaf=2, random_state=21)
rf_best_avg, rf_best_std = evaluate_model_cv(rf_best, X_train, y_train, n_splits=10)

RF max_depth=22, n_estimators=100, min_samples_leaf=2
train -  0.94312   |   valid -  0.85926
train -  0.94477   |   valid -  0.91111
train -  0.94477   |   valid -  0.84444
train -  0.94312   |   valid -  0.85926
train -  0.94064   |   valid -  0.88889
train -  0.94806   |   valid -  0.87407
train -  0.94394   |   valid -  0.90370
train -  0.93899   |   valid -  0.85185
train -  0.94316   |   valid -  0.86567
train -  0.94481   |   valid -  0.85821
Average accuracy on crossval is 0.87165
Std is 0.02131
CPU times: user 1.06 s, sys: 3.96 ms, total: 1.07 s
Wall time: 1.06 s


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [51]:
best_model = RandomForestClassifier(n_estimators=50, max_depth=22, random_state=21).fit(X_train, y_train)

In [52]:
y_pred = best_model.predict(X_test)
print(f"Итоговая accuracy на тестовом датасете: {accuracy_score(y_test, y_pred)}")

Итоговая accuracy на тестовом датасете: 0.9171597633136095


In [53]:
days = {0: 'Понедельник', 1: 'Вторник', 2: 'Среда', 3: 'Четверг',
        4: 'Пятница', 5: 'Суббота', 6: 'Воскресенье'}

error_analysis = pd.DataFrame({
    'y_true': y_test.values,
    'y_pred': y_pred
})
error_analysis['error'] = (error_analysis['y_true'] != error_analysis['y_pred']).astype(int)

print("Анализ ошибок по дням недели:")
print("-" * 55)
for day_num in sorted(y_test.unique()):
    mask = error_analysis['y_true'] == day_num
    total = mask.sum()
    errors = error_analysis.loc[mask, 'error'].sum()
    error_pct = errors / total * 100
    print(f"{days[day_num]:12s} (day {day_num}): {errors:3d} ошибок из {total} значений = {error_pct:.1f}%")

worst_day = error_analysis.groupby('y_true')['error'].apply(lambda x: x.sum() / len(x) * 100).idxmax()
print(f"\nЧаще всего модель ошибается (в %) в отношении дня недели: {days[worst_day]} (день {worst_day})")

Анализ ошибок по дням недели:
-------------------------------------------------------
Понедельник  (day 0):   7 ошибок из 27 значений = 25.9%
Вторник      (day 1):   7 ошибок из 55 значений = 12.7%
Среда        (day 2):   3 ошибок из 30 значений = 10.0%
Четверг      (day 3):   3 ошибок из 80 значений = 3.8%
Пятница      (day 4):   3 ошибок из 21 значений = 14.3%
Суббота      (day 5):   4 ошибок из 54 значений = 7.4%
Воскресенье  (day 6):   1 ошибок из 71 значений = 1.4%

Чаще всего модель ошибается (в %) в отношении дня недели: Понедельник (день 0)


In [54]:
joblib.dump(best_model, 'best_model.sav')

['best_model.sav']